# Data Mining 1 — Friendfeed: account types and best posting times

**Team:** Gustav Axelsson, Andreas Johansson, Leo Hansson

This notebook runs the full two-stage pipeline on the Friendfeed (Version 2) dataset.

**Stage 1 — who posts (clustering).** Roll the six raw tables up to one row per user, then cluster accounts into behavioural types (broadcaster, quiet follower, conversation-driven, ...).

**Stage 2 — when to post (timing).** Go back to one row per post, attach each author's cluster label, and find which posting times give the best engagement *for each cluster*.

The key link between the stages is the **cluster label**: it is produced in Stage 1 and joined onto posts in Stage 2. Same raw data, two grains.

> Run the cells top to bottom. The only thing you must edit is the **Config** cell (Section 1): the data folder, the delimiter, and whether the files have a header row. After loading, check the previews before continuing.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# seaborn is preinstalled on Colab; installed here only if missing
try:
    import seaborn as sns
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn"])
    import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Setup complete.")


## 1. Load the data

The Friendfeed files ship **without a documented header** and the text fields can contain the delimiter and newlines, so loading is the fragile step. This cell centralises everything you might need to change.

**How to get the files into Colab** (uncomment one option in the next cell):
- upload them by hand, or
- mount your Google Drive and point `DATA_DIR` at the folder.

The column names below come straight from the dataset codebook, applied **by position**. Note two codebook quirks handled here: the second `NumImg`/`ImgURL` pair is really **NumVid/VidURL**, and `GeoX`, `GeoY`, `Reserved` are empty.


In [ ]:
# ============================ CONFIG — EDIT THIS CELL ============================
DATA_DIR = "."          # folder holding the CSV files
SEP      = ","          # try "\t" or ";" if the preview looks wrong
HAS_HEADER = False      # True if the first row of each file is column names
ENCODING = "utf-8"      # try "latin-1" if you get a UnicodeDecodeError

# File names — change to match your files
FILES = {
    "entries":   "entries.csv",
    "comments":  "comments.csv",
    "likes":     "likes.csv",
    "users":     "users.csv",
    "following": "following.csv",
    "services":  "services.csv",
}

# Column order from the codebook (used when HAS_HEADER = False, or to rename by position)
SCHEMA = {
    "entries":   ["PostID","PostedBy","SourceName","SourceURL","GeoX","GeoY",
                  "Timestamp","Text","NumImg","ImgURL","NumVid","VidURL"],
    "comments":  ["PostID","EntryID","PostedBy","SourceName","SourceURL","GeoX","GeoY",
                  "Timestamp","Text","NumImg","ImgURL","NumVid","VidURL"],
    "likes":     ["userID","PostID","Timestamp"],
    "users":     ["ID","Type","Name","Reserved","Description"],
    "following": ["FollowedID","FollowerID"],   # Version 2 order
    "services":  ["UserID","ServiceID","ServiceName","ServiceURL",
                  "UserNameOnService","UserURLOnService"],
}
# ================================================================================
print("Config set. DATA_DIR =", DATA_DIR)


In [ ]:
# --- Option A: upload files by hand (uncomment to use) ---
# from google.colab import files
# uploaded = files.upload()
# DATA_DIR = "."

# --- Option B: mount Google Drive (uncomment to use) ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/friendfeed"   # <-- point at your folder


In [ ]:
def load_table(key):
    path = os.path.join(DATA_DIR, FILES[key])
    names = SCHEMA[key]
    if HAS_HEADER:
        df = pd.read_csv(path, sep=SEP, encoding=ENCODING, dtype=str,
                         quotechar='"', on_bad_lines="skip")
    else:
        df = pd.read_csv(path, sep=SEP, encoding=ENCODING, header=None, dtype=str,
                         quotechar='"', on_bad_lines="skip",
                         names=names + [f"extra_{i}" for i in range(0, 5)])
        df = df[[c for c in names if c in df.columns]]
    # keep only documented columns; assign canonical names by position where possible
    df = df.iloc[:, :len(names)]
    df.columns = names[:df.shape[1]]
    return df

raw = {}
for k in FILES:
    try:
        raw[k] = load_table(k)
        print(f"{k:10s} loaded  shape={raw[k].shape}")
    except Exception as e:
        print(f"{k:10s} FAILED  -> {e}")


In [ ]:
# PREVIEW — check that columns line up before continuing.
# If the text is spilling across the wrong columns, fix SEP / HAS_HEADER above and re-run.
for k in raw:
    print("="*80)
    print(k.upper())
    display(raw[k].head(3))


## 2. Clean and standardise

Drop the empty fields (`GeoX`, `GeoY`, `Reserved`), rename the video columns, parse the GMT+1 timestamps, and coerce the count columns to numbers. Everything downstream uses short canonical names so the rest of the notebook does not depend on the raw headers.


In [ ]:
def to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)

def to_dt(s):
    return pd.to_datetime(s, errors="coerce")   # timestamps are GMT+1 (local)

# ---- ENTRIES ----
e = raw["entries"].copy()
entries = pd.DataFrame({
    "post_id": e["PostID"].astype(str),
    "author":  e["PostedBy"].astype(str),
    "source":  e.get("SourceName"),
    "ts":      to_dt(e["Timestamp"]),
    "text":    e.get("Text").fillna("") if "Text" in e else "",
    "num_img": to_num(e["NumImg"]) if "NumImg" in e else 0,
    "num_vid": to_num(e["NumVid"]) if "NumVid" in e else 0,   # 2nd codebook pair = video
})

# ---- COMMENTS ----
c = raw["comments"].copy()
comments = pd.DataFrame({
    "comment_id": c["PostID"].astype(str),
    "entry_id":   c["EntryID"].astype(str),
    "author":     c["PostedBy"].astype(str),
    "ts":         to_dt(c["Timestamp"]),
})

# ---- LIKES ----
l = raw["likes"].copy()
likes = pd.DataFrame({
    "user_id": l["userID"].astype(str),
    "post_id": l["PostID"].astype(str),
    "ts":      to_dt(l["Timestamp"]),
})

# ---- USERS ----
u = raw["users"].copy()
users = pd.DataFrame({
    "id":   u["ID"].astype(str),
    "type": u.get("Type"),
    "description": u.get("Description").fillna("") if "Description" in u else "",
})

# ---- FOLLOWING ----
f = raw["following"].copy()
following = pd.DataFrame({
    "followed_id": f["FollowedID"].astype(str),
    "follower_id": f["FollowerID"].astype(str),
})

# ---- SERVICES ----
s = raw["services"].copy()
services = pd.DataFrame({"user_id": s["UserID"].astype(str)})

print("entries  ", entries.shape)
print("comments ", comments.shape)
print("likes    ", likes.shape)
print("users    ", users.shape)
print("following", following.shape)
print("services ", services.shape)
print("\nEntries date range:", entries['ts'].min(), "->", entries['ts'].max())


## 3. Stage 1 — build the user table

One row per user, with features grouped into the three behavioural dimensions from the proposal.

- **Production** — how much and what they post (`num_entries`, `num_comments_made`, `comment_to_entry_ratio`, media/external shares, active days).
- **Network** — who follows them (`num_followers`, `num_following`, `follower_following_ratio`).
- **Reception** — how others react (`likes_received`, `comments_received`, per-entry averages, distinct audience).
- **Profile** — `has_description`, `num_services`.

These are chosen to separate the account types we expect. A broadcaster shows high entries, high followers, a high follower/following ratio, and a low comment ratio. A quiet follower shows the reverse. A conversation-driven user shows a high comment-to-entry ratio.


In [ ]:
# ---- per-entry engagement (needed for both stages) ----
likes_per_entry    = likes.groupby("post_id").size().rename("likes")
comments_per_entry = comments.groupby("entry_id").size().rename("comments")

entries = entries.join(likes_per_entry, on="post_id").join(comments_per_entry, on="post_id")
entries["likes"] = entries["likes"].fillna(0)
entries["comments"] = entries["comments"].fillna(0)
entries["engagement"] = entries["likes"] + entries["comments"]

# ---- PRODUCTION ----
prod = entries.groupby("author").agg(
    num_entries      = ("post_id", "size"),
    active_days      = ("ts", lambda x: x.dt.normalize().nunique()),
    first_post       = ("ts", "min"),
    last_post        = ("ts", "max"),
    share_with_image = ("num_img", lambda x: (x > 0).mean()),
    share_with_video = ("num_vid", lambda x: (x > 0).mean()),
    share_external   = ("source", lambda x: x.notna().mean()),
    num_sources      = ("source", "nunique"),
)
prod["span_days"] = (prod["last_post"] - prod["first_post"]).dt.days.clip(lower=0)
num_comments_made = comments.groupby("author").size().rename("num_comments_made")

# ---- NETWORK ----
num_followers = following.groupby("followed_id").size().rename("num_followers")
num_following = following.groupby("follower_id").size().rename("num_following")

# ---- RECEPTION ----
recv = entries.groupby("author").agg(
    likes_received    = ("likes", "sum"),
    comments_received = ("comments", "sum"),
)
# breadth of audience (distinct likers of a user's entries)
likers = likes.merge(entries[["post_id", "author"]], on="post_id", how="inner")
num_distinct_likers = likers.groupby("author")["user_id"].nunique().rename("num_distinct_likers")

# ---- PROFILE ----
users["has_description"] = (users["description"].str.len() > 0).astype(int)
num_services = services.groupby("user_id").size().rename("num_services")
print("Feature blocks computed.")


In [ ]:
# ---- assemble one row per user ----
uid = users.set_index("id")
U = pd.DataFrame(index=uid.index)
U = (U.join(prod).join(num_comments_made).join(num_followers).join(num_following)
       .join(recv).join(num_distinct_likers).join(num_services))

count_cols = ["num_entries","num_comments_made","num_followers","num_following",
              "likes_received","comments_received","num_distinct_likers",
              "num_services","active_days","span_days","num_sources"]
U[count_cols] = U[count_cols].fillna(0)
share_cols = ["share_with_image","share_with_video","share_external"]
U[share_cols] = U[share_cols].fillna(0)

# derived ratios (the +1 avoids divide-by-zero and softens tiny denominators)
U["comment_to_entry_ratio"]  = U["num_comments_made"] / (U["num_entries"] + 1)
U["follower_following_ratio"] = U["num_followers"]    / (U["num_following"] + 1)
U["avg_likes_per_entry"]      = U["likes_received"]    / (U["num_entries"] + 1)
U["avg_comments_per_entry"]   = U["comments_received"] / (U["num_entries"] + 1)
U["has_description"]          = uid["has_description"]
U["type"]                     = uid["type"]

print("User table:", U.shape)
U.head()


In [ ]:
# ---- choose the clustering population ----
# Users/Following are a static Oct snapshot; activity is Aug-Sep, so many users are
# completely inactive in the window. Cluster only accounts with some activity, so the
# result is about behavioural TYPES rather than one giant "inactive" blob.
ACTIVE_ONLY = True
U["total_activity"] = U["num_entries"] + U["num_comments_made"]

if ACTIVE_ONLY:
    Uc = U[U["total_activity"] > 0].copy()
else:
    Uc = U.copy()

print(f"Total users: {len(U)}   |   used for clustering: {len(Uc)}")


## 4. Stage 1 — cluster the accounts

Count features are heavily right-skewed (a power law), so we `log1p` them, then standardise to z-scores so no single feature dominates the k-means distance. We pick `k` with the elbow (inertia) and the silhouette score, then profile each cluster to name it.


In [ ]:
CLUSTER_FEATURES = [
    "num_entries", "num_comments_made", "comment_to_entry_ratio",
    "active_days", "share_with_image", "share_with_video", "share_external",
    "num_followers", "num_following", "follower_following_ratio",
    "avg_likes_per_entry", "avg_comments_per_entry", "num_distinct_likers",
    "num_services", "has_description",
]
LOG_FEATURES = [
    "num_entries", "num_comments_made", "active_days",
    "num_followers", "num_following", "num_distinct_likers",
    "avg_likes_per_entry", "avg_comments_per_entry", "num_services",
]

X = Uc[CLUSTER_FEATURES].copy()
for col in LOG_FEATURES:
    X[col] = np.log1p(X[col])

Xz = StandardScaler().fit_transform(X)
print("Feature matrix for clustering:", Xz.shape)


In [ ]:
# ---- choose k ----
ks = range(2, 9)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xz)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xz, km.labels_, sample_size=min(10000, len(Xz)), random_state=42))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(list(ks), inertias, "o-"); ax[0].set_title("Elbow (inertia)"); ax[0].set_xlabel("k")
ax[1].plot(list(ks), sils, "o-");     ax[1].set_title("Silhouette");      ax[1].set_xlabel("k")
plt.tight_layout(); plt.show()
print("Silhouette by k:", {k: round(s, 3) for k, s in zip(ks, sils)})


In [ ]:
# ---- fit final model ----
K = 4   # <-- set from the plots above (start where silhouette peaks / the elbow bends)
km = KMeans(n_clusters=K, n_init=25, random_state=42).fit(Xz)
Uc["cluster"] = km.labels_

print("Cluster sizes:")
print(Uc["cluster"].value_counts().sort_index())


In [ ]:
# ---- profile clusters on the raw (pre-transform) features to interpret them ----
profile = Uc.groupby("cluster")[CLUSTER_FEATURES].median()
profile["size"] = Uc["cluster"].value_counts().sort_index()
display(profile.round(2))

# Heatmap of z-scored cluster means makes the personality of each cluster pop out.
means_z = Uc.groupby("cluster")[CLUSTER_FEATURES].mean()
means_z = (means_z - means_z.mean()) / means_z.std()
plt.figure(figsize=(11, 0.6 * len(CLUSTER_FEATURES)))
sns.heatmap(means_z.T, cmap="RdBu_r", center=0, annot=True, fmt=".1f", cbar_kws={"label": "z-score vs other clusters"})
plt.title("Cluster fingerprints (red = high for this cluster)")
plt.xlabel("cluster"); plt.tight_layout(); plt.show()


In [ ]:
# ---- name the clusters ----
# Read the fingerprint heatmap and fill in labels, e.g.:
#   high entries + high followers + low comment ratio  -> "Broadcaster"
#   low entries  + high following + low followers      -> "Quiet follower"
#   high comment_to_entry_ratio                        -> "Conversationalist"
CLUSTER_NAMES = {i: f"Cluster {i}" for i in range(K)}
# CLUSTER_NAMES = {0: "Broadcaster", 1: "Quiet follower", 2: "Conversationalist", 3: "..."}

Uc["cluster_name"] = Uc["cluster"].map(CLUSTER_NAMES)
Uc[["num_entries","num_followers","comment_to_entry_ratio","avg_likes_per_entry","cluster_name"]].head()


In [ ]:
# ---- optional: PCA scatter to eyeball separation ----
p2 = PCA(n_components=2, random_state=42).fit_transform(Xz)
plt.figure(figsize=(7, 6))
sns.scatterplot(x=p2[:, 0], y=p2[:, 1], hue=Uc["cluster_name"].values, s=10, alpha=0.5, linewidth=0)
plt.title("Accounts in 2D (PCA)"); plt.xlabel("PC1"); plt.ylabel("PC2")
plt.legend(markerscale=2, fontsize=8); plt.tight_layout(); plt.show()


## 5. Stage 2 — attach the cluster label to posts

Back to one row per entry. Each post already has its engagement and timestamp; now we join the author's cluster label on by `author`, and add the timing features.

**The normalisation that matters.** Broadcasters get far more engagement than quiet followers no matter when they post, so raw likes-per-hour would just re-measure who is popular. We divide each post's engagement by its **author's own average**, so a value above 1 means the post beat that author's norm. Averaging that ratio across time slots isolates the *time* effect.


In [ ]:
posts = entries.dropna(subset=["ts"]).copy()

# join cluster label from Stage 1 (only authors that were clustered)
author_cluster = Uc["cluster_name"]
posts["cluster_name"] = posts["author"].map(author_cluster)
posts = posts.dropna(subset=["cluster_name"])

# timing features (local GMT+1)
posts["hour"]    = posts["ts"].dt.hour
posts["dow"]     = posts["ts"].dt.dayofweek          # 0 = Monday
posts["dow_name"] = posts["ts"].dt.day_name()

# author-normalised engagement
author_mean = posts.groupby("author")["engagement"].transform("mean")
posts["norm_engagement"] = posts["engagement"] / (author_mean + 1e-9)

print("Posts with a cluster label:", len(posts))
posts[["author","cluster_name","hour","dow_name","engagement","norm_engagement"]].head()


## 6. Stage 2 — best posting times per cluster

For each cluster we build an hour-of-day by day-of-week grid of average normalised engagement. Cells with very few posts are noisy, so we mask any slot below a minimum post count before reading off recommendations.


In [ ]:
MIN_POSTS_PER_CELL = 20   # raise/lower depending on how much data each cluster has
DOW_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

clusters = sorted(posts["cluster_name"].unique())
fig, axes = plt.subplots(len(clusters), 1, figsize=(12, 3.2 * len(clusters)))
if len(clusters) == 1: axes = [axes]

for ax, cl in zip(axes, clusters):
    sub = posts[posts["cluster_name"] == cl]
    grid  = sub.pivot_table(index="dow_name", columns="hour", values="norm_engagement", aggfunc="mean")
    count = sub.pivot_table(index="dow_name", columns="hour", values="norm_engagement", aggfunc="size")
    grid  = grid.reindex(DOW_ORDER)
    count = count.reindex(DOW_ORDER)
    grid  = grid.mask(count < MIN_POSTS_PER_CELL)     # hide thin cells
    sns.heatmap(grid, ax=ax, cmap="viridis", cbar_kws={"label": "norm. engagement"})
    ax.set_title(f"{cl}  (n = {len(sub)} posts)")
    ax.set_xlabel("hour of day (GMT+1)"); ax.set_ylabel("")
plt.tight_layout(); plt.show()


In [ ]:
# ---- recommended posting windows per cluster (top slots with enough support) ----
rows = []
for cl in clusters:
    sub = posts[posts["cluster_name"] == cl]
    g = sub.groupby(["dow_name","hour"]).agg(
        norm_engagement=("norm_engagement","mean"),
        n=("norm_engagement","size"),
    ).reset_index()
    g = g[g["n"] >= MIN_POSTS_PER_CELL].sort_values("norm_engagement", ascending=False)
    top = g.head(5).copy()
    top.insert(0, "cluster", cl)
    rows.append(top)

recommendations = pd.concat(rows, ignore_index=True)
recommendations["norm_engagement"] = recommendations["norm_engagement"].round(2)
print("Best posting slots per cluster (normalised engagement, 1.0 = author's own average):")
display(recommendations)


## 7. Export results

Save the user table with cluster labels and the posting recommendations so you can drop them into the report.


In [ ]:
U_out = U.copy()
U_out["cluster_name"] = Uc["cluster_name"]   # NaN for users not clustered (inactive)
U_out.to_csv("users_with_clusters.csv")
recommendations.to_csv("posting_recommendations.csv", index=False)
profile.round(3).to_csv("cluster_profiles.csv")
print("Wrote: users_with_clusters.csv, posting_recommendations.csv, cluster_profiles.csv")

# In Colab, download them:
# from google.colab import files
# for fn in ["users_with_clusters.csv","posting_recommendations.csv","cluster_profiles.csv"]:
#     files.download(fn)


## Notes and honest caveats (worth a paragraph in the report)

- **Loading is the risk.** If any preview looked misaligned, fix `SEP` / `HAS_HEADER` first. A wrong delimiter silently corrupts every feature downstream.
- **Static vs windowed data.** Followers/following/services are an Oct snapshot; posts are Aug–Sep. `follower_following_ratio` is a fixed snapshot, not a within-window behaviour. Fine to use, worth stating.
- **Why normalise engagement.** Without the author-baseline division, the timing heatmaps would mostly rank clusters by popularity, not reveal good hours. State this explicitly, it is the crux of Stage 2.
- **"Best time" = when the audience is awake.** The recommendation captures when a cluster's followers are active. That is exactly the actionable thing, but it is correlation, not a controlled experiment.
- **Leakage.** Stage 2 uses only time + author-cluster to explain engagement, no feature is derived from the likes/comments being predicted except the target itself. If you later add a model, keep it that way, and consider training on August, testing on September.
- **Extensions.** Add content features (media, text length, hashtags) to a regression that also controls for cluster and time; or feed the cluster label into that model to test whether account type changes the payoff of the same content.
